# SG-DMNMF — Stability-Guided Deep Motif NMF for Community Detection

Colab companion for the CSET347 minor project (Arpit Sahu, S24BCAU0195, Team 8).

This notebook **does not change the research code**. It runs the existing project, reproduces
the quick experiment, and displays the generated tables and figures.

### Step 0 — get the project into Colab (choose one)

**Option A (Drive):** copy the `sg-dmnmf` folder to `My Drive`, then run the next cell.

**Option B (Upload):** zip the project folder on your computer, then run the upload cell instead.

In [ ]:
# Option A: Google Drive (default)
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/sg-dmnmf'
assert os.path.isdir(PROJECT), 'Upload the sg-dmnmf folder to My Drive first.'
print('Project:', PROJECT)

In [ ]:
# Option B: upload a zip instead (run this INSTEAD of the Drive cell above)
# from google.colab import files
# import zipfile, os
# uploaded = files.upload()
# zip_name = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_name) as z:
#     z.extractall('/content')
# PROJECT = '/content/sg-dmnmf'
# print(os.listdir(PROJECT))

## 1. Install dependencies and import the project

In [ ]:
import subprocess, sys, os
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', os.path.join(PROJECT, 'requirements.txt')])
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)
print('Ready. Python', sys.version.split()[0])

In [ ]:
from src.data import load_karate
from src.models import DMRALite, SymNMF
from src.evaluate import evaluate_all
from src.motif import count_triangles

A, y, name = load_karate()
k = len(set(y))
print(f'{name}: n={A.shape[0]}, edges={int(A.sum())//2}, k={k}, triangles={count_triangles(A)}')

sym = evaluate_all(y, SymNMF(k=k, seed=42).fit(A))
deep = evaluate_all(y, DMRALite(k=k, alpha=0.5, max_iter=150, seed=42).fit(A))
print(f'SymNMF    ACC={sym["ACC"]:.3f}  NMI={sym["NMI"]:.3f}')
print(f'DMRA-lite ACC={deep["ACC"]:.3f}  NMI={deep["NMI"]:.3f}')

## 2. Reproduce the quick experiment

This creates `results/` and `figures/` inside the project folder, exactly like
`python experiments/run_all.py --quick` on your machine.

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, 'experiments/run_all.py', '--quick'],
                     cwd=PROJECT, capture_output=True, text=True)
print('exit code:', res.returncode)
print(res.stdout[-2500:])
if res.returncode != 0:
    print(res.stderr[-2000:])

## 3. Generated results

In [ ]:
import pandas as pd
e1 = pd.read_csv(os.path.join(PROJECT, 'results', 'e1_clean_benchmark.csv'))
pd.set_option('display.max_rows', 100)
cols = ['Dataset', 'Method', 'ACC_mean', 'NMI_mean', 'ARI_mean', 'AMI_mean', 'Modularity_mean']
display(e1[cols].round(3))

In [ ]:
e2 = pd.read_csv(os.path.join(PROJECT, 'results', 'e2_alpha_sensitivity.csv'))
metric_col = e2['Metric'].iloc[0]
show = e2[e2['Metric'] == metric_col][['Dataset', 'Alpha', 'Mean', 'Stability']]
display(show.round(3))

## 4. Figures produced by the experiment

In [ ]:
from IPython.display import Image, display
import glob
figs = sorted(glob.glob(os.path.join(PROJECT, 'figures', 'e2_alpha_sensitivity_*.png')))
for f in figs:
    print(os.path.basename(f))
    display(Image(filename=f, width=700))

In [ ]:
figs = sorted(glob.glob(os.path.join(PROJECT, 'figures', 'e3_robustness_*_edge_deletion.png')))
for f in figs:
    print(os.path.basename(f))
    display(Image(filename=f, width=700))

## 5. Reading the results

- **E1** compares SG-DMNMF with SymNMF, DANMF, DMoN-lite, Leiden, fixed α, no-motif and a
  label-using oracle. `DMRA-stab(a=...)` is the proposed label-free rule.
- **E2** shows how clustering quality and perturbation stability change with α. The selected α
  is the peak of the stability curve, chosen without labels.
- **E3** shows robustness under edge deletion/addition and triangle-targeted deletion.
- **E4** removes each model component to measure its contribution.

Datasets marked **offline synthetic** or **unlabeled** are exactly as in the local project:
synthetic stand-ins are never presented as real data, and the real Dolphin graph stays
unlabeled when no authentic community attribute exists.

For the written report, use the Overleaf files in `overleaf/`.